In [1]:
from pyod.models.cblof import CBLOF
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import os
import pickle

from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt 
%matplotlib inline

import sys
sys.path.append('..')
sys.path.append('../..')

from src.utils import *
from src.Components.data_processing import data_process

In [2]:
file_path = '../../datasets/Dodgers/101-freeway-traffic.test.out'

columns = ['value', 'anomaly']

df = pd.read_csv(file_path, names=columns, header=None)

In [3]:
X_train, X_test = train_test_split(df, test_size=0.3, shuffle=False)
train_x = X_train[['value']][X_train['anomaly']== 0]
train_y = X_train[['anomaly']].values.ravel()
test_data = X_test[['value']]
gtruth = X_test[['anomaly']]

In [4]:
model = CBLOF(random_state=42)
model.fit(train_x)

/home/rojan/anaconda3/envs/venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


CBLOF(alpha=0.9, beta=5, check_estimator=False, clustering_estimator=None,
   contamination=0.1, n_clusters=8, n_jobs=None, random_state=42,
   use_weights=False)

### Evaluating the Model

In [6]:
anomaly_scores = model.decision_function(test_data)
y_pred = model.predict(test_data)
model.threshold_

2.880646817248511

In [18]:
aa = X_test['value'].iloc[50:51].values.reshape(-1, 1)
aa

array([[40]])

In [19]:
model.decision_function(aa)

array([1.42857143])

In [51]:
thres = raw_thresholds(anomaly_scores, contamination=0.062)
thres = thres - 0.65
thres

2.7068882365272082

In [52]:
print(np.max(anomaly_scores))
print(np.min(anomaly_scores))
print(np.mean(anomaly_scores))

31.428571428571104
0.06932367149753205
1.879116905176533


In [53]:
thres_np = [1 if x > thres else 0 for x in anomaly_scores]
thres_np

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 1,
 1,


In [54]:
prec = precision_score(gtruth, thres_np)
recall = recall_score(gtruth, thres_np)
f1 = f1_score(gtruth, thres_np)

print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precision Score: 0.3630  Recall: 0.5015  f1_score: 0.4212


In [55]:
prec = precision_score(gtruth, y_pred)
recall = recall_score(gtruth, y_pred)
f1 = f1_score(gtruth, y_pred)

print(f'Precision Score: {prec:.4f}  Recall: {recall:.4f}  f1_score: {f1:.4f}')

Precision Score: 0.3904  Recall: 0.4783  f1_score: 0.4299


In [57]:
file_name = f'../../saved_models/clof_dodgers_v2.sav'

pickle.dump(model, open(file_name, 'wb'))